# PHISHGUARD AI — Dataset Audit and EDA

This notebook uses only the `URL` and `label` columns from the UCI PhiUSIIL CSV. It does not use page/HTML-derived source features and never visits target URLs.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px

sys.path.insert(0, str(Path.cwd() / 'src'))
from phishguard.data.dataset import read_dataset
from phishguard.features.url_features import extract_features_frame

DATASET = Path('../data/raw/PhiUSIIL_Phishing_URL_Dataset.csv')
if not DATASET.exists():
    DATASET = Path('data/raw/PhiUSIIL_Phishing_URL_Dataset.csv')
print('Dataset path:', DATASET.resolve())

In [ ]:
dataset, audit = read_dataset(DATASET)
print(audit)
dataset.head()

## Class distribution

Internal convention: `1 = phishing`, `0 = legitimate`. The source UCI convention is the opposite (`0 = phishing`, `1 = legitimate`) and is normalized by the loader.

In [ ]:
class_counts = dataset['target'].map({0: 'Legitimate', 1: 'Phishing'}).value_counts().rename_axis('Class').reset_index(name='Records')
px.pie(class_counts, names='Class', values='Records', hole=0.55, title='URL class distribution')

## URL-only feature engineering

Features are recomputed from the URL string. The source's precomputed web-page attributes are not used.

In [ ]:
sample = dataset.sample(min(10000, len(dataset)), random_state=42)
features = extract_features_frame(sample['url'])
features['target'] = sample['target'].to_numpy()
features.describe().T

In [ ]:
features.corr(numeric_only=True).round(2)

## Domain-aware evaluation note

Use `make_domain_splits` from `phishguard.data.splitting` for model experiments. Do not use a random row split as the primary estimate because URLs from one registered domain can appear in multiple rows.

In [ ]:
from phishguard.data.splitting import make_domain_splits
splits = make_domain_splits(dataset['target'], dataset['domain_group'])
splits.summary